In [1]:
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict
import csv
from datetime import datetime, timezone, timedelta
from openlocationcode import openlocationcode as olc
import os
import json
import random
import numpy as np
from ast import literal_eval

In [2]:
datafold = 'NYC'  # 这里设置数据集名称
stage = "sft"
sidfile = "Dec-12-2025_cvq_semitic_codes"

In [ ]:
# 生成训练序列和测试序列    uid,pid,time
def generate_train_sequences(df: pd.DataFrame, window_size: int, step_size: int, mask_prob: float) -> pd.DataFrame:

    df = df.copy()
    df['time'] = pd.to_datetime(df['time'])

    results = []

    for uid, group in df.groupby('uid'):
        group = group.sort_values('time').reset_index(drop=True)
        
        if len(group) > 80:
            group = group.iloc[-80:]
        
        n = len(group)

        if n < window_size:
            if n >= 10:  
                input_pids = group['pid'].iloc[:-1].tolist()
                input_times = group['time'].iloc[:-1].tolist()
                target_pid = group['pid'].iloc[-1]
                target_time = group['time'].iloc[-1]

                results.append({
                    'uid': uid,
                    'pids': input_pids,
                    'times': input_times,
                    'target': target_pid,
                    'target_time': target_time
                })
            continue 

        for start in range(n - 1, window_size - 2, -step_size):

            window = group.iloc[start - window_size + 1 : start + 1]

            input_pids = window['pid'].iloc[:-1].tolist()
            input_times = window['time'].iloc[:-1].tolist()
            target_pid = window['pid'].iloc[-1]
            target_time = window['time'].iloc[-1]

            results.append({
                'uid': uid,
                'pids': input_pids,
                'times': input_times,
                'target': target_pid,
                'target_time': target_time
            })

    train = pd.DataFrame(results)

    train['times'] = train['times'].apply(lambda x: [t.strftime('%Y-%m-%d %H:%M') for t in x])
    train['target_time'] = train['target_time'].dt.strftime('%Y-%m-%d %H:%M')

    return train

def generate_test_sequences(df: pd.DataFrame, window_size: int):

    df = df.copy()
    df['time'] = pd.to_datetime(df['time'])

    results = []

    for uid, group in df.groupby('uid'):
        group = group.sort_values('time').reset_index(drop=True)
        n = len(group)

        if n < window_size:
            if n > 1:
                results.append({
                    'uid': uid,
                    'pids': group['pid'].iloc[:-1].tolist(),
                    'times': group['time'].iloc[:-1].tolist(),
                    'target': group['pid'].iloc[-1],
                    'target_time': group['time'].iloc[-1]
                })
            continue
        window = group.iloc[n - window_size:]
        results.append({
            'uid': uid,
            'pids': window['pid'].iloc[:-1].tolist(),
            'times': window['time'].iloc[:-1].tolist(),
            'target': window['pid'].iloc[-1],
            'target_time': window['time'].iloc[-1]
        })

    test = pd.DataFrame(results)

    test['times'] = test['times'].apply(lambda x: [t.strftime('%Y-%m-%d %H:%M') for t in x])
    test['target_time'] = test['target_time'].dt.strftime('%Y-%m-%d %H:%M')

    return test

# 使用示例
if __name__ == "__main__":

    train_df = pd.read_csv(f"{datafold}/train_data.csv")
    val_df = pd.read_csv(f"{datafold}/valid_data.csv")
    test_df = pd.read_csv(f"{datafold}/test_data.csv")

    window_size = 50
    step_size = 10
    mask_prob = 0.1

    train = generate_train_sequences(train_df, window_size, step_size, mask_prob)
    train.to_csv(f"{datafold}/train_sequences.csv", index=False)

    val = generate_test_sequences(val_df, window_size)
    test = generate_test_sequences(test_df, window_size)
    val.to_csv(f"{datafold}/valid_sequences.csv", index=False)
    test.to_csv(f"{datafold}/test_sequences.csv", index=False)

    print("训练、验证和测试序列已生成并保存。")

In [22]:
# 生成 LLM 训练样本 json 格式
def encode_item(item):
    if not item:
        return ""
    labels = ['a', 'b', 'c', 'd']  # 最多4维
    if len(item) > 4:
        raise ValueError(f"Item 维度超过4: {item}")
    return "".join(f"<{labels[i]}_{val}>" for i, val in enumerate(item))

def load_pid_to_code(codebook_path):
    df_code = pd.read_csv(codebook_path)
    pid2code = {}
    for _, row in df_code.iterrows():
        pid = row['pid']
        semitic_str = row['sid']
        try:
            item_list = literal_eval(semitic_str)
            code_str = encode_item(item_list)
            pid2code[pid] = code_str
        except Exception as e:
            print(f"跳过 pid={pid}, sid={semitic_str}: {e}")
    return pid2code


def sequence_to_llm_sample(uid, seq, times, target, target_time, pid2code):
    if len(seq) < 2:
        return None
    try:
        codes = [pid2code[pid] for pid in seq]
    except KeyError as e:
        print(f"警告: pid {e} 不在码本中，跳过该样本")
        return None

    history_parts = []
    for code, t in zip(codes, times):
        history_parts.append(f"{t} visited {code}")

    # 拼接历史访问记录
    history_str = ", ".join(history_parts)

    # 生成完整输入文本
    input_text = (
        f"User_{uid} checkin history: {history_str}.\n"
        f"When {target_time} user_{uid} is likely to visit:"
    )

    return {
        "instruction": "Here is a record of a user's POI accesses, your task is based on the history to predict the POI that the user is likely to access at the specified time.",
        "input": input_text,
        "output": pid2code[target]
    }

def generate_llm_json(expanded_df, pid2code, output_json_path):
    samples = []
    for _, row in expanded_df.iterrows():
        uid = row['uid']
        seq = row['pids']
        times = row['times']
        target = row['target']
        target_time = row['target_time']
        sample = sequence_to_llm_sample(uid, seq, times, target, target_time, pid2code)
        if sample is not None:
            samples.append(sample)
    
    # 写入 JSON 数组文件
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)
    
    print(f"已生成 {len(samples)} 个 sid based LLM 训练样本，保存至: {output_json_path}")
    return samples

# ===== 主程序 =====
if __name__ == "__main__":
    
    # 加载码本
    pid2code = load_pid_to_code(f"{datafold}/{sidfile}.csv")

    # 检查目录是否存在
    if not os.path.exists(f"{datafold}/{stage}/{sidfile}"):
        os.makedirs(f"{datafold}/{stage}/{sidfile}")

    # train
    df_expand = pd.read_csv(f"{datafold}/train_sequences.csv")
    df_expand['pids'] = df_expand['pids'].apply(literal_eval)
    df_expand['times'] = df_expand['times'].apply(literal_eval)
    output_path = f"{datafold}/{stage}/{sidfile}/llm_train.json"  
    generate_llm_json(df_expand, pid2code, output_path)

    # valid
    df_expand = pd.read_csv(f"{datafold}/valid_sequences.csv")
    df_expand['pids'] = df_expand['pids'].apply(literal_eval)
    df_expand['times'] = df_expand['times'].apply(literal_eval)
    output_path = f"{datafold}/{stage}/{sidfile}/llm_valid.json"
    generate_llm_json(df_expand, pid2code, output_path)

    # test
    df_expand = pd.read_csv(f"{datafold}/test_sequences.csv")
    df_expand['pids'] = df_expand['pids'].apply(literal_eval)
    df_expand['times'] = df_expand['times'].apply(literal_eval)
    output_path = f"{datafold}/{stage}/{sidfile}/llm_test.json"
    generate_llm_json(df_expand, pid2code, output_path)


已生成 2848 个 sid based LLM 训练样本，保存至: NYC/sft/Dec-07-2025_vq_semitic_codes/llm_train.json
已生成 841 个 sid based LLM 训练样本，保存至: NYC/sft/Dec-07-2025_vq_semitic_codes/llm_valid.json
已生成 791 个 sid based LLM 训练样本，保存至: NYC/sft/Dec-07-2025_vq_semitic_codes/llm_test.json


In [10]:
# id_based
# 生成 LLM 训练样本 json 格式


def sequence_to_llm_sample(uid, seq, times, target, target_time):
    if len(seq) < 2:
        return None

    history_parts = []
    for pid, t in zip(seq, times):
        history_parts.append(f"{t} visited {pid}")

    # 拼接历史访问记录
    history_str = ", ".join(history_parts)

    # 生成完整输入文本
    input_text = (
        f"User_{uid} checkin history: {history_str}.\n"
        f"When {target_time} user_{uid} is likely to visit:"
    )

    return {
        "instruction": "Here is a record of a user's POI accesses, your task is based on the history to predict the POI that the user is likely to access at the specified time.",
        "input": input_text,
        "output": f"{target}"
    }

def generate_llm_json(expanded_df, output_json_path):
    samples = []
    for _, row in expanded_df.iterrows():
        uid = row['uid']
        seq = row['pids']
        times = row['times']
        target = row['target']
        target_time = row['target_time']
        sample = sequence_to_llm_sample(uid, seq, times, target, target_time)
        if sample is not None:
            samples.append(sample)
    
    # 写入 JSON 数组文件
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)
    
    print(f"已生成 {len(samples)} 个 id based LLM 训练样本，保存至: {output_json_path}")
    return samples

# ===== 主程序 =====
if __name__ == "__main__":
    
    # 检查目录是否存在
    if not os.path.exists(f"{datafold}/{stage}/{sidfile}"):
        os.makedirs(f"{datafold}/{stage}/{sidfile}")

    # train
    df_expand = pd.read_csv(f"{datafold}/train_sequences.csv")
    df_expand['pids'] = df_expand['pids'].apply(literal_eval)
    df_expand['times'] = df_expand['times'].apply(literal_eval)
    output_path = f"{datafold}/{stage}/{sidfile}/llm_id_train.json"  
    generate_llm_json(df_expand, output_path)

    # valid
    df_expand = pd.read_csv(f"{datafold}/valid_sequences.csv")
    df_expand['pids'] = df_expand['pids'].apply(literal_eval)
    df_expand['times'] = df_expand['times'].apply(literal_eval)
    output_path = f"{datafold}/{stage}/{sidfile}/llm_id_valid.json"
    generate_llm_json(df_expand, output_path)

    # test
    df_expand = pd.read_csv(f"{datafold}/test_sequences.csv")
    df_expand['pids'] = df_expand['pids'].apply(literal_eval)
    df_expand['times'] = df_expand['times'].apply(literal_eval)
    output_path = f"{datafold}/{stage}/{sidfile}/llm_id_test.json"
    generate_llm_json(df_expand, output_path)



已生成 7308 个 id based LLM 训练样本，保存至: TKY/sft/Dec-12-2025_cvq_semitic_codes/llm_id_train.json
已生成 1988 个 id based LLM 训练样本，保存至: TKY/sft/Dec-12-2025_cvq_semitic_codes/llm_id_valid.json
已生成 1906 个 id based LLM 训练样本，保存至: TKY/sft/Dec-12-2025_cvq_semitic_codes/llm_id_test.json
